# Tutorial 9: EXAFS from an amorphous ensemble

EXAFS measures the local structure around one chosen element: the fine structure
χ(k) above an absorption edge, and its Fourier transform |χ(R)|, which shows the
neighbour shells around the absorbing atom. It is element-selective where a total
scattering measurement is not, so it is a second, independent check of an
amorphous model. This tutorial computes EXAFS from an ensemble at the three levels
AmorphGen provides, which answer different questions:

| Level | What it gives | Needs |
|---|---|---|
| 1. Geometric descriptors (`--exafs`) | N, R, σ², C₃ of the first shell, as an EXAFS paper reports them | base package |
| 2. Harmonic-fit diagnostic (`--harmonic-fit`) | the N, R, σ² a Gaussian EXAFS fit would report for the ensemble | base package |
| 3. FEFF spectra (`--feff-write`, `--feff-run`) | χ(k) and \|χ(R)\| averaged over absorber sites and structures | an external FEFF executable |

**Data**: the a-Ga₂O₃ ensembles shipped with Tutorial 8 (400 atoms, MACE-MPA-0),
read from `../T8_validation_workflow/structures/`; this tutorial adds no data.

**FEFF** (sections 3–5) is not part of AmorphGen. Build FEFF8.5L from
[xraypy/feff85exafs](https://github.com/xraypy/feff85exafs) (BSD licence) and
point `FEFF_EXECUTABLE` at it, or put `feff8l` on `PATH`. Without it, sections 1–3
run and the FEFF-running cells say so and stop. With FEFF, the tutorial makes 40
short FEFF runs, about a minute on four cores.

The full reference is the EXAFS guide, `docs/guides/exafs.md`.

In [ ]:
# Run this notebook from its own tutorial directory (Tutorials/T9_exafs/).
import os
import shutil

import matplotlib.pyplot as plt
import numpy as np

from amorphgen.analysis import StructureAnalyser

T8 = "../T8_validation_workflow/structures"
ensembles = {name: StructureAnalyser(f"{T8}/{name}") for name in ("relaxed", "annealed")}
colour = {"relaxed": "tab:blue", "annealed": "tab:red"}
for name, an in ensembles.items():
    print(f"{name:9s}: {len(an.atoms_list)} structures of {len(an.atoms_list[0])} atoms")

FEFF = (os.environ.get("FEFF_EXECUTABLE") or shutil.which("feff8l")
        or shutil.which("feff"))
print("FEFF executable:", FEFF or "not found -- sections 4 and 5 will be skipped")

---
## 1. Geometric descriptors: the numbers an EXAFS paper reports

An EXAFS fit reports, for each shell around the absorber, a coordination number N,
a distance R, a disorder σ² and sometimes the third cumulant C₃ that measures
asymmetry. `exafs_descriptors()` computes the same quantities directly from the
geometry. Pairs are **ordered**: `"Ga-O"` means Ga absorber and O scatterer, the
first shell seen at the Ga K edge. Errors come from resampling whole structures,
so they describe the ensemble, not the pooled atoms.

`R_eff` is the mean distance weighted the way EXAFS weights it: by 1/R² and by the
photoelectron mean free path. In an asymmetric shell, which is what an amorphous
oxide has, it sits below the plain mean.

In [ ]:
fig, ax = plt.subplots(figsize=(6, 3.6))
for name, an in ensembles.items():
    d = an.exafs_descriptors("Ga-O")["Ga-O"]
    print(f"{name:9s} N = {d['N']:.2f} ± {d['N_err']:.2f}   R = {d['R']:.4f} Å   "
          f"R_eff = {d['R_eff']:.4f} Å   σ² = {d['sigma2']:.4f} Å²   "
          f"C3 = {d['C3']:.5f} Å³   skew = {d['skew']:.2f}")
    ax.plot(d["P_R"]["r"], d["P_R"]["p"], color=colour[name], label=name)
ax.set(xlabel="Ga–O distance (Å)", ylabel="P(R), first shell", title="Ga–O first-shell distribution")
ax.legend(frameon=False)
fig.tight_layout()
plt.show()

Both ensembles have 4.5 O around each Ga at a mean 1.93 Å, with σ² ≈ 0.009 Å². The
distributions have a long tail on the far side, skewness 1.6–1.7, so `R_eff` (1.92 Å)
sits about 0.01 Å below the mean. These are geometric quantities: an EXAFS fit to the
same structures would not return them, as section 2 shows.

```bash
amorphgen --analyse --input-dir ../T8_validation_workflow/structures/relaxed \
    --exafs Ga-O --save-plot analysis
```
writes `analysis_exafs.csv` with every value and its error; `--exafs` alone covers
all cation–anion pairs.

---
## 2. What a Gaussian EXAFS fit would report

Most published EXAFS fits model each shell as a Gaussian. Fitted to an asymmetric
shell, a Gaussian returns a lower N, a shorter R and a smaller σ² than the structure
has. `harmonic_fit()` measures that bias for the ensemble: it fits a Gaussian shell to
the ensemble's own first-shell EXAFS signal over the same k range a paper would use,
weighted by a bundled FEFF6L backscattering amplitude. It is an approximation to a
full FEFF fit, but it shows which quantity to compare against a published table.

In [ ]:
for name, an in ensembles.items():
    h = an.harmonic_fit("Ga-O", kmin=3.0, kmax=12.0)["Ga-O"]
    print(f"{name} (k = {h['kmin']:g}-{h['kmax']:g} Å⁻¹, fit quality: {h['fit_quality']})")
    print(f"   geometric: N = {h['N_geom']:.2f}  R = {h['R_geom']:.3f} Å  σ² = {h['sigma2_geom']:.4f} Å²")
    print(f"   fit:       N = {h['N_fit']:.2f}  R = {h['R_fit']:.3f} Å  σ² = {h['sigma2_fit']:.4f} Å²")
    print(f"   bias:      N {h['N_bias_percent']:+.1f} %   R {h['R_bias']:+.3f} Å   "
          f"σ² {h['sigma2_bias_percent']:+.0f} %")

A Gaussian fit to these structures would report N ≈ 4.1 instead of 4.5 (−9 %), R
0.025 Å short and about half the σ². So a measured Ga–O N of 4.1 from such a fit is
consistent with this model, not a contradiction of it. Compare a published fit table
with the `fit` values. Set `kmin` and `kmax` to the paper's range, because the bias
grows with `kmax`.

```bash
amorphgen --analyse --input-dir ../T8_validation_workflow/structures/relaxed \
    --harmonic-fit Ga-O --harmonic-kmin 3 --harmonic-kmax 12 --save-plot analysis
```
In a reference YAML, `exafs: {source: harmonic-fit, Ga-O: {N_expected: [...]}}`
scores the ensemble against a published fit in the same way.

---
## 3. FEFF input: one cluster per absorber site

Levels 1 and 2 use only the first shell. FEFF computes the whole spectrum, including
the outer shells and multiple scattering, from a cluster cut around each absorbing
atom. The ensemble average then needs many such calculations: one per absorber site,
averaged first within each structure and then over structures with equal weight,
before any Fourier transform.

`write_feff_jobs()` writes one FEFF input per sampled site, with periodic images
filled in, plus a `manifest.json` that fingerprints every input. Here: Ga sites, five
per structure, in four relaxed cells. `stoichiometry="structure"` gives FEFF the
composition of the whole cell for its potentials, which is what a periodic solid
needs (the default, `"cluster"`, lets FEFF infer it from the cluster).

In [ ]:
from amorphgen.analysis.feff import write_feff_jobs

cells = ensembles["relaxed"].atoms_list[:4]
manifest = write_feff_jobs(cells, "Ga", "feff_ga", n_sites=5, radius=6.0,
                           stoichiometry="structure")
print(manifest["n_jobs"], "jobs:", [job["dir"] for job in manifest["jobs"][:5]], "...")
first = os.path.join("feff_ga", manifest["jobs"][0]["dir"], "feff.inp")
print("".join(open(first).readlines()[:30]))

The header carries the FEFF cards. `S02 1.0` leaves the amplitude unreduced, since
this is a forward model, not a fit. RPATH follows the cluster radius. The
potentials list the absorbing Ga separately from the other elements, with the cell's
composition. Writing the same inputs again changes nothing; writing different ones
into the same directory is refused, so old spectra can never be attached to new
structures. The command-line equivalent:

```bash
amorphgen --analyse --input-dir ../T8_validation_workflow/structures/relaxed \
    --feff-write feff_ga --feff-absorber Ga --feff-sites 5 --feff-stoichiometry structure
```

---
## 4. Run FEFF, average and transform: one call

`feff_pipeline()` writes the trees, runs FEFF on several jobs at once, collects the
output and saves the figures and CSV tables. Jobs that already finished with matching
inputs are kept, so an interrupted run is simply started again. A tree with a failed
job is not averaged, because a partial average leans towards whichever sites ran.

In [ ]:
from amorphgen.analysis.feff import feff_pipeline, feff_uncertainty

if FEFF is None:
    print("No FEFF executable: set FEFF_EXECUTABLE to run sections 4 and 5.")
else:
    out = feff_pipeline(cells, "Ga", "feff_ga", executable=FEFF, workers=4,
                        n_sites=5, radius=6.0, stoichiometry="structure",
                        kmin=3.0, kmax=12.0, plot_dir="analysis", prefix="t9")
    print(out["Ga"]["run"])
    ga = out["Ga"]["result"]
    print(out["Ga"]["report"])

    chik, chir = feff_uncertainty(ga)
    fig, axes = plt.subplots(1, 2, figsize=(12, 3.8))
    axes[0].plot(ga["k"], ga["chi"] * ga["k"] ** 2, color="tab:blue")
    axes[0].fill_between(ga["k"], chik["bootstrap_low"], chik["bootstrap_high"],
                         color="tab:blue", alpha=0.2, lw=0, label="95% bootstrap over structures")
    axes[0].set(xlabel="k (Å⁻¹)", ylabel="k²χ(k) (Å⁻²)", xlim=(0, ga["k"].max()))
    axes[1].plot(ga["r"], ga["chir_mag"], color="tab:blue")
    axes[1].fill_between(ga["r"], chir["bootstrap_low"], chir["bootstrap_high"],
                         color="tab:blue", alpha=0.2, lw=0)
    axes[1].axvline(1.93, ls=":", color="0.5", label="mean Ga–O distance")
    axes[1].set(xlabel="R (Å), phase-uncorrected", ylabel="|χ(R)| (Å⁻³)", xlim=(0, 6))
    for ax in axes:
        ax.legend(frameon=False)
    fig.tight_layout()
    plt.show()

The 20 FEFF runs take a few seconds. The first |χ(R)| peak sits near 1.4 Å, about
0.5 Å below the 1.93 Å Ga–O distance of section 1. That is the scattering phase shift,
not an error; most experimental figures are plotted the same way ("phase-uncorrected").
The band is the spread between the four structures. Four is far too few, and the
report says the average is not converged.

`analysis/` now holds `t9_Ga_feff_chik` and `t9_Ga_feff_chir` (`.png` and `.csv`),
plus `_uncertainty` and `_per_structure` tables, in the style of the other analysis
plots. On the command line, writing, running and collecting is one command, and
several absorbers each get their own tree:

```bash
amorphgen --analyse --input-dir ../T8_validation_workflow/structures/relaxed \
    --feff-write feff_ga --feff-absorber Ga --feff-sites 5 --feff-stoichiometry structure \
    --feff-run --feff-executable "$FEFF_EXECUTABLE" --feff-workers 4 \
    --feff-kmin 3 --feff-kmax 12 --save-plot analysis
```

---
## 5. Is it converged? Path length, sampling and comparison with experiment

A FEFF average has three settings to converge before it is compared with anything:

- **Path length and filters.** RPATH sets the longest scattering path; FEFF's default
  path filters also drop weak long paths. `--feff-card` (or `cards=`) sets both.
- **Absorber sampling**: more sites per structure until the average stops changing.
- **Structure sampling**: more independent structures until the band is small enough.

Here the same 20 sites are recalculated with a 10 Å cluster, paths up to 8 Å and
looser filters, and the two averages are compared over R ≤ 4 Å and over the whole
k range.

In [ ]:
from amorphgen.analysis.feff import chi_fourier_transform

if FEFF is None:
    print("No FEFF executable: section 5 skipped.")
else:
    long = feff_pipeline(cells, "Ga", "feff_ga_long", executable=FEFF, workers=4,
                         n_sites=5, radius=10.0, stoichiometry="structure",
                         cards={"RPATH": "8", "CRITERIA": "1.0 0.5"},
                         kmin=3.0, kmax=12.0)["Ga"]["result"]

    def transform(res):
        ft = chi_fourier_transform(res["k"], res["chi"], kweight=2, kmin=3, kmax=12, dk=1)
        return ft["r"], ft["chir_re"] + 1j * ft["chir_im"]

    r, short_ft = transform(ga)
    _, long_ft = transform(long)
    near = r <= 4.0
    d_r = np.linalg.norm((short_ft - long_ft)[near]) / np.linalg.norm(long_ft[near])
    k = ga["k"]
    window = (k >= 3) & (k <= 12)
    d_k = (np.linalg.norm(((ga["chi"] - long["chi"]) * k ** 2)[window])
           / np.linalg.norm((long["chi"] * k ** 2)[window]))
    print(f"difference over R ≤ 4 Å: {100 * d_r:.1f} %    over k = 3-12 Å⁻¹: {100 * d_k:.1f} %")

    fig, ax = plt.subplots(figsize=(6, 3.6))
    ax.plot(r, np.abs(short_ft), label="6 Å cluster, default paths")
    ax.plot(r, np.abs(long_ft), "--", label="10 Å cluster, paths to 8 Å")
    ax.axvspan(4, 8, color="0.92", lw=0)
    ax.set(xlabel="R (Å), phase-uncorrected", ylabel="|χ(R)| (Å⁻³)", xlim=(0, 8))
    ax.legend(frameon=False)
    fig.tight_layout()
    plt.show()

Below 4 Å the two averages differ by about 2 %; over the full k range by about 12 %,
almost all of it from long paths beyond 4 Å (grey). This is what a full a-Ga₂O₃ study
found too: the first shells converge with modest settings, while the long-range part
keeps changing with path length. State which R range a comparison covers, and test
path length, absorber count and structure count for your own system. That study used
all 160 Ga sites in each of 20 cells to meet a 2 % criterion on R ≤ 4 Å, and the
number of independent cells was still its largest uncertainty.

The longer calculation from the command line; `--feff-card` is repeatable and takes
any FEFF card:

```bash
amorphgen --analyse --input-dir ../T8_validation_workflow/structures/relaxed \
    --feff-write feff_ga_long --feff-absorber Ga --feff-sites 5 --feff-radius 10 \
    --feff-stoichiometry structure --feff-card RPATH=8 --feff-card "CRITERIA=1.0 0.5" \
    --feff-run --feff-executable "$FEFF_EXECUTABLE" --feff-workers 4 \
    --feff-kmin 3 --feff-kmax 12 --save-plot analysis_long
```

**Comparing with experiment.** Use the same k-weight, k window and taper as the
measurement, and check whether the published |χ(R)| is phase-corrected. Leave S0²,
an energy shift E₀ and extra damping as explicit, reported nuisance parameters. A
static model has no thermal disorder, so its outer shells come out too sharp.
`examples/exafs/ga2o3_validation.py` does this for a published a-Ga₂O₃ spectrum.

---
## Going further

- **A cluster** (one FEFF run per site soon reaches thousands): write the trees with
  `--feff-write`, run each `joblist.txt` as a Slurm array
  (`python -m amorphgen.analysis.feff --jobs-dir . --job "$DIR"`), then collect with
  `--feff-collect DIR`. A directory of per-element trees is collected tree by tree.
- **Several edges**: `--feff-absorber In,Ga,Zn` writes one tree per absorber,
  `DIR/In`, `DIR/Ga`, `DIR/Zn`, and plots each as `analysis_<El>_feff_*`.
- **YAML**: the same settings go in the `analysis:` block (`feff_write`,
  `feff_absorber`, `feff_run`, `feff_executable`, `feff_workers`, `feff_cards`).
- **Your own ensemble**: point the cells above at the `random_opt/` of Tutorial 2 or 3
  or the `final/` of Tutorial 5.
- **The guide**: `docs/guides/exafs.md` covers provenance, imported results, potential
  composition and FEFF's 15,000-path limit.